# Model Trainging – Heart Disease Dataset
*Training and evaluation of models for Cardio - Risk Prediction project*  

---

## Table of Contents


<a id='imports'></a>
## Reproducibility & Imports  
---

In [30]:
# Reproducibility
import os, sys, numpy as np
import joblib

# Imports
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))

# Models & tools
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from scripts.modeltraining.utils import evaluate_classification


SEED = 42

In [10]:
df = joblib.load("../outputs/dim_red/df_mix.pkl")
df

,PCA_1,tSNE_1,UMAP_1,DEATH_EVENT
0,0.198919,-2.080552,3.357985,0
1,1.188811,-3.854919,4.325813,0
2,-0.825730,0.951819,2.561738,0
3,0.408272,-2.558243,3.499924,0
4,1.784139,-4.937725,5.562924,0
...,...,...,...,...
303,1.358504,-4.887924,4.780306,1
304,-0.288906,5.934488,0.704528,1
305,0.483580,-1.383888,3.780901,1
306,0.118853,-2.241611,4.726557,1


In [11]:
df_copy = df.copy()
TARGET_COL = 'DEATH_EVENT'

X = df_copy.drop(columns=[TARGET_COL])
y = df_copy[TARGET_COL]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Data split

In [12]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Split shapes:")
print("  X_tr:", X_tr.shape, "| X_te:", X_te.shape)

Split shapes:
  X_tr: (246, 3) | X_te: (62, 3)


<a id='random-forest'></a>
---
## Random Forest

In [ ]:
params_grid = {
    'n_estimators': [800, 900, 1000],
    'max_depth': [5, 6, 7],
    'min_samples_split': [4, 6, 8],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 0.4, 0.6],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced'],
}

In [38]:
grid = GridSearchCV(
    estimator = RandomForestClassifier(n_jobs=-1, random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best parameters:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

# save the best model
os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/rf.joblib')

Fitting 5 folds for each of 2304 candidates, totalling 11520 fits
Best parameters: {'bootstrap': True, 'class_weight': None, 'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 900}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8920
2,Precision (pos=1),0.8571
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9268
5,F1,0.8571
6,F0.5,0.8571
7,F2,0.8571
8,ROC-AUC,0.9396
9,PR-AUC (Average Precision),0.8590



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,3,18


['../outputs/models/rf.joblib']

<a id='adaboost'></a>
## AdaBoost

In [39]:
params_grid = {
    'n_estimators': [50, 100, 200, 300, 600],
    'learning_rate': [0.01, 0.05, 0.1],
    'estimator__max_depth': [1, 2, 3, 4],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__class_weight': [None, 'balanced'],
}

In [40]:
grid = GridSearchCV(
    estimator = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=SEED), random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/ada.joblib')

Fitting 5 folds for each of 360 candidates, totalling 1800 fits
Best params: {'estimator__class_weight': None, 'estimator__max_depth': 3, 'estimator__min_samples_leaf': 3, 'learning_rate': 0.05, 'n_estimators': 200}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8920
2,Precision (pos=1),0.8571
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9268
5,F1,0.8571
6,F0.5,0.8571
7,F2,0.8571
8,ROC-AUC,0.9547
9,PR-AUC (Average Precision),0.9036



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,3,18


['../outputs/models/ada.joblib']

<a id='xgboost'></a>
## XGBoost


In [13]:
params_grid = {
    'n_estimators': [50, 100, 150, 200],   
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_child_weight': [1, 2, 4, 6, 7],
    'subsample': [0.5, 1.0],
    'reg_lambda': [0.01, 0.05, 0.5],
}


In [14]:
xgb = XGBClassifier(
    tree_method = 'hist',
    objective = 'binary:logistic',
    eval_metric = 'logloss',
    n_jobs = -1,
    random_state = SEED,
)

grid = GridSearchCV(
    estimator = xgb,
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/xgb.joblib')

Fitting 5 folds for each of 1080 candidates, totalling 5400 fits
Best params: {'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 6, 'n_estimators': 200, 'reg_lambda': 0.5, 'subsample': 1.0}


,Metric,Value
0,Accuracy,0.8387
1,Balanced accuracy,0.8084
2,Precision (pos=1),0.7895
3,Recall / Sensitivity (TPR),0.7143
4,Specificity (TNR),0.9024
5,F1,0.7500
6,F0.5,0.7732
7,F2,0.7282
8,ROC-AUC,0.9297
9,PR-AUC (Average Precision),0.8078



Confusion matrix:


,Pred 0,Pred 1
Actual 0,37,4
Actual 1,6,15


['../outputs/models/xgb.joblib']

## Naive Bayes

In [26]:
params_grid = {
    "var_smoothing": np.logspace(-12, -6, 13),
}

In [27]:
grid = GridSearchCV(
    estimator= GaussianNB(),
    param_grid=params_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    n_jobs=-1,
    verbose=10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs("../outputs/models", exist_ok=True)
joblib.dump(model, "../outputs/models/naive_bayes.joblib")

Fitting 5 folds for each of 13 candidates, totalling 65 fits
Best params: {'var_smoothing': np.float64(1e-12)}


,Metric,Value
0,Accuracy,0.8710
1,Balanced accuracy,0.8792
2,Precision (pos=1),0.7600
3,Recall / Sensitivity (TPR),0.9048
4,Specificity (TNR),0.8537
5,F1,0.8261
6,F0.5,0.7851
7,F2,0.8716
8,ROC-AUC,0.9129
9,PR-AUC (Average Precision),0.8578



Confusion matrix:


,Pred 0,Pred 1
Actual 0,35,6
Actual 1,2,19


['../outputs/models/naive_bayes.joblib']

## SVM

In [42]:
params_grid = {
    "kernel": ["rbf", "linear", "poly"],
    "C": [0.1, 1, 5, 7, 10],
    "gamma": ["scale", "auto"],
    "class_weight": [None, "balanced"],
    "degree": [2, 3, 4],

}

In [43]:
grid = GridSearchCV(
    estimator = SVC(probability=True, random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = "accuracy",
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/svm.joblib')


Fitting 5 folds for each of 180 candidates, totalling 900 fits
Best params: {'C': 1, 'class_weight': None, 'degree': 3, 'gamma': 'auto', 'kernel': 'poly'}


,Metric,Value
0,Accuracy,0.8710
1,Balanced accuracy,0.8444
2,Precision (pos=1),0.8421
3,Recall / Sensitivity (TPR),0.7619
4,Specificity (TNR),0.9268
5,F1,0.8000
6,F0.5,0.8247
7,F2,0.7767
8,ROC-AUC,0.9268
9,PR-AUC (Average Precision),0.8597



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,5,16


['../outputs/models/svm.joblib']

## Logistic Classifier

In [ ]:
params_grid = {
    "penalty": [None, "l2"],
    "C": np.logspace(-3, 3, 7),             # inverse of regularization strength
    "class_weight": [None, "balanced"],
}

In [41]:
grid = GridSearchCV(
    estimator = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = "accuracy",                       # or "balanced_accuracy"
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print("Best params:", grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs("../outputs/models", exist_ok=True)
joblib.dump(model, "../outputs/models/logistic.joblib")

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Best params: {'C': np.float64(0.1), 'class_weight': 'balanced', 'penalty': 'l2'}


,Metric,Value
0,Accuracy,0.8548
1,Balanced accuracy,0.8670
2,Precision (pos=1),0.7308
3,Recall / Sensitivity (TPR),0.9048
4,Specificity (TNR),0.8293
5,F1,0.8085
6,F0.5,0.7600
7,F2,0.8636
8,ROC-AUC,0.9106
9,PR-AUC (Average Precision),0.8508



Confusion matrix:


,Pred 0,Pred 1
Actual 0,34,7
Actual 1,2,19


['../outputs/models/logistic.joblib']